# 05 — Limpieza bibliográfica

Esta fase parte exclusivamente de `autores_unam_normalizados.csv`, resultado del
código 04. Conserva todas las filas, su orden y las **15 columnas intermedias**,
incluida `Fuente_origen`. Los archivos limpios antiguos no son entradas del proceso.

Se mantienen intactos `Fuente_origen`, `indice`, `Autor_norm`, `Afiliacion1`,
`Afiliacion2` y `Area`. No se deduplica, no se fusiona, no se vuelve a filtrar
UNAM, no se completan vacíos ni se copian metadatos entre filas. `SubArea` queda vacía.

**Insumos**

- `../04_Limpieza/02_normalizacion/autores_unam_normalizados.csv`
- `../04_Limpieza/03_limpieza_bibliografica/casos_revision_bibliografica_resueltos.csv`
- `../04_Limpieza/03_limpieza_bibliografica/segmentacion_isbn_validada.csv`

La revisión resuelta conserva las diez columnas históricas y añade `Doi`,
`Alcance`, `Aplicar` y `URL_evidencia`. Cada decisión se restringe a su valor original
y al contexto indicado. Los alcances son `CELDA`, `ELEMENTO` (keyword completo),
`COLA` (fragmento terminal exacto del abstract) y `SEPARADOR` (lista de keywords ya
inspeccionada). No utiliza automáticamente la revisión histórica de 753 filas.
Las sustituciones de abstracts son reparaciones locales, nunca resúmenes nuevos.

La tabla ISBN conserva los mismos dígitos y documenta rango, grupo, registrante,
fecha y fuente de cada segmentación. Los rangos proceden del archivo `isbn.dat`
de python-stdnum, que identifica la exportación de la International ISBN Agency
con fecha **4 de enero de 2026** y serial `6e5a8502-5e3f-4baa-9b1a-ff835dd18851`.
La evidencia queda congelada en el CSV: no se descarga ni se cambia en cada ejecución.
Esto comprueba formato y checksum, no la asignación del ISBN a la publicación.
Un identificador válido que no tenga segmentación documentada pasa a revisión.
No se requiere instalar python-stdnum; el código utiliza pandas y la biblioteca estándar.

**Resultados, en `../04_Limpieza/03_limpieza_bibliografica/`**

- `autores_unam_limpios.csv`: base completa, únicamente si no hay errores bloqueantes.
- `casos_revision_bibliografica.csv`: errores y avisos, siempre con sus valores originales.
- `auditoria_limpieza_bibliografica.csv`: conteos y huellas de los tres insumos.
- `auditoria_detallada_limpieza_bibliografica.csv`: una entrada por celda modificada.

`actualizar_archivos = False` crea archivos nuevos o verifica salidas existentes
idénticas. Para reemplazar resultados diferentes, resguardar primero la versión
anterior en GitHub Desktop y cambiar expresamente a `True`. Esta autorización
nunca permite sobrescribir la entrada ni los archivos de decisiones.
Todos los CSV se releen como texto y se comparan exactamente después de guardarlos.

Los casos de severidad `ERROR` bloquean la base final, sin borrar un campo para
hacerlo pasar la validación. Se revisan en el CSV de casos y se documenta la decisión
en el archivo resuelto, únicamente cuando pertenece a limpieza. Los `AVISO`
conservan el contenido y no bloquean la salida; por ejemplo, un abstract que ya
llegue truncado no se reconstruye durante esta fase. Los avisos de truncamiento
no se resuelven copiando otro abstract. DOI y URL reciben validación sintáctica,
no comprobaciones de registro ni de disponibilidad HTTP.

Ejecutar secuencialmente desde un kernel reiniciado en VS Code. Las funciones,
validaciones y ejecución se mantienen en una sola celda de código, como en la
versión anterior, con apartados comentados.


In [1]:
import os
import re
import csv
import html
import json
import hashlib
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from urllib.parse import urlsplit
import pandas as pd

# ============================================================
# 05 - LIMPIEZA BIBLIOGRÁFICA
# ============================================================
# La entrada es la salida del código 04, nunca una base histórica limpia.
# No se cambia ninguna identidad, afiliación, índice, fuente o área.
# No se completan vacíos, no se consultan publicaciones por internet,
# no se propagan campos entre filas, no se fusionan ni eliminan registros.
# Los guiones ISBN proceden exclusivamente de la tabla de segmentación
# documentada. No hay máscaras de longitudes fijas ni conversión a float.

# ============================================================
# 1. RUTAS Y CONFIGURACIÓN
# ============================================================

def buscar_raiz_repo():
    inicio = Path.cwd().resolve()
    for candidato in [inicio, *inicio.parents]:
        if (candidato / '04_Limpieza').is_dir() and (candidato / 'notebooks').is_dir():
            return candidato
    raise FileNotFoundError('Abre en VS Code la carpeta del repositorio Tesis_Multimodelo.')

RAIZ_REPO = buscar_raiz_repo()
archivo_entrada = RAIZ_REPO / '04_Limpieza' / '02_normalizacion' / 'autores_unam_normalizados.csv'
carpeta_salida = RAIZ_REPO / '04_Limpieza' / '03_limpieza_bibliografica'
archivo_revision = carpeta_salida / 'casos_revision_bibliografica_resueltos.csv'
archivo_segmentacion = carpeta_salida / 'segmentacion_isbn_validada.csv'
archivo_salida = carpeta_salida / 'autores_unam_limpios.csv'
archivo_pendientes = carpeta_salida / 'casos_revision_bibliografica.csv'
archivo_auditoria = carpeta_salida / 'auditoria_limpieza_bibliografica.csv'
archivo_detalle = carpeta_salida / 'auditoria_detallada_limpieza_bibliografica.csv'

# False: crea salidas nuevas o verifica las existentes idénticas.
# True: autoriza reemplazar SOLO estas cuatro salidas, nunca los insumos.
actualizar_archivos = True
# Opcional: fijar la huella de un checkpoint específico. No obliga a 5106 filas.
sha256_entrada_esperada = ''

COLUMNAS = ['Fuente_origen', 'indice', 'Titulo', 'Año', 'Autor_norm',
            'Afiliacion1', 'Afiliacion2', 'ISBN', 'ISSN', 'Doi', 'URL',
            'Area', 'SubArea', 'Keywords', 'Abstract']
PROTEGIDAS = ['Fuente_origen', 'indice', 'Autor_norm', 'Afiliacion1', 'Afiliacion2', 'Area']
MODIFICABLES = ['Titulo', 'Año', 'ISBN', 'ISSN', 'Doi', 'URL', 'Keywords', 'Abstract', 'SubArea']
AREAS_VALIDAS = {'CC', 'IA', 'ISBD', 'RS', 'SIAV', 'TC'}
COLUMNAS_REVISION = ['Fuente_origen', 'indice', 'Titulo', 'Campo', 'Valor_original',
                    'Problema', 'Accion_recomendada', 'Decision_manual', 'Valor_final',
                    'Comentario_resolucion', 'Doi', 'Alcance', 'Aplicar', 'URL_evidencia']
COLUMNAS_PENDIENTES = ['Fila_entrada', 'Fuente_origen', 'indice', 'Titulo', 'Doi', 'Campo',
                      'Valor_original', 'Valor_propuesto', 'Problema', 'Severidad',
                      'Accion_recomendada', 'Decision_manual', 'Valor_final', 'Comentario_resolucion']
COLUMNAS_DETALLE = ['Fila_entrada', 'Fuente_origen', 'indice', 'Autor_norm', 'Titulo', 'Doi',
                   'Campo', 'Valor_anterior', 'Valor_nuevo', 'Reglas', 'Casos_manuales', 'Evidencia']
SCI_RE = re.compile(r'(?i)(?<![A-Za-z0-9])[+-]?[0-9]+(?:[.,][0-9]+)?\s*[eE]\s*[+-]?[0-9]+(?![A-Za-z0-9])')
DOI_RE = re.compile(r'^10\.[0-9]{4,9}(?:\.[0-9]+)*/[^\s\x00-\x1f\x7f]+$')
ENTIDAD_RE = re.compile(r'&(?:#[0-9]+|#x[0-9a-fA-F]+|[A-Za-z][A-Za-z0-9]+);')
ASCII_MINUSCULAS = str.maketrans('ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz')
GUIONES = str.maketrans({c: '-' for c in '‐‑‒–—−'})
AUSENCIA_ABSTRACT = {'[no abstract available]', 'no abstract available',
                    '[abstract not available]', 'abstract not available', '[no abstract]'}
PREFIJO_ABSTRACT_RE = re.compile(r'^\s*(?:Abstract|PARAPHRASED SUMMARY)\s*:\s*', re.I)
ARTEFACTOS_RE = re.compile(r'\ufffd|ĝ\.,•|!antification|con!ict|quanti#cation|e"ective|di"erent')

# Lista cerrada de grafías técnicas; nunca se baja a minúsculas el resto
# del término ni se cambia sistemáticamente Title Case por sentence case.
GRAFIAS_TECNICAS = ['UNAM', 'IIMAS', 'CNN', 'IoT', 'LLM', 'DNA', 'RNA', 'ResNet',
                   'srsRAN', 'pH', '5G', 'EEG', 'ECG', 'EMG', 'MRI', 'NLP', 'SVM',
                   'RAG', 'LSTM', 'BERT', 'GAN', 'PSNR', 'SSIM', 'DICOM', 'SPECT',
                   'NIfTI', 'FLAIR', 'CPU', 'GPU', 'AI', 'OpenPGP', 'MongoDB']
TECNICAS = {x.casefold(): x for x in GRAFIAS_TECNICAS}
GENERIC_ALLCAPS_SINGLE = {'CONVERGENCE', 'ASTROCYTES', 'NETWORK', 'COMMUNICATION',
    'SUBSTITUTION', 'TRANSPORTATION', 'SEARCH', 'ALGORITHMS', 'SELECTION',
    'BENCHMARKING', 'PLACEMENT', 'RECEPTORS'}
GENERIC_ALLCAPS_PHRASES = {'SWARM OPTIMIZATION ALGORITHM', 'KEY GENETIC ALGORITHM',
    'ROUTING PROBLEM', 'AVERAGED HAUSDORFF DISTANCE', 'EVOLUTIONARY ALGORITHM',
    'SUBSET-SELECTION', 'BATTERY ENERGY-STORAGE'}
EV_HYPHEN_EXCEPTIONS = ['E - learning', 'Image processing - methods']

# ============================================================
# 2. FUNCIONES GENERALES: TEXTO, HTML Y ERRORES DE CODIFICACIÓN
# ============================================================

class RevisionNecesaria(ValueError):
    def __init__(self, codigo, mensaje):
        self.codigo = codigo
        super().__init__(mensaje)


def nfc(s):
    return unicodedata.normalize('NFC', s)


def quitar_espacios(s):
    return re.sub(r'\s+', ' ', s).strip()


def texto_clave(s):
    # Solo para emparejar una corrección ya documentada con SU valor original.
    return quitar_espacios(nfc(s))


def doi_clave(s):
    s = nfc(s).strip()
    for _ in range(3):
        nuevo = re.sub(r'^(?:https?://(?:dx\.)?doi\.org/|doi\s*:\s*)', '', s, flags=re.I).strip()
        if nuevo == s:
            break
        s = nuevo
    return s.translate(ASCII_MINUSCULAS)


def codificaciones_reversibles():
    resultado = {}
    # Solo secuencias completas inequívocas, nunca sustituir Ã/Â/â aisladas.
    for c in 'áéíóúÁÉÍÓÚñÑüÜ¿¡©±µ×αβεγΔΩ–—‘’“”\u00a0':
        for encoding in ('latin1', 'cp1252'):
            try:
                malo = c.encode('utf-8').decode(encoding)
            except UnicodeError:
                continue
            if malo != c and len(malo) >= 2:
                resultado[malo] = c
    return resultado

MOJIBAKE = codificaciones_reversibles()
SUB_MAP = str.maketrans('0123456789+-=()aeoxhklmnpstijruv',
                        '₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎ₐₑₒₓₕₖₗₘₙₚₛₜᵢⱼᵣᵤᵥ')
SUP_MAP = str.maketrans('0123456789+-=()ni', '⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁼⁽⁾ⁿⁱ')


def convertir_indice_html(m):
    tipo, contenido = m.group(1).lower(), m.group(2)
    contenido = re.sub(r'</?(?:i|b|em|strong|span)\b[^>]*>', '', contenido, flags=re.I)
    tabla = SUB_MAP if tipo == 'sub' else SUP_MAP
    if contenido and all(ord(c) in tabla for c in contenido):
        return contenido.translate(tabla)
    # Cuando Unicode no basta, mantener explícitamente la jerarquía matemática.
    return ('_(' if tipo == 'sub' else '^(') + contenido + ')'


def clean_text(s, eventos):
    if not isinstance(s, str):
        raise TypeError('Todos los campos se leen como texto.')
    original = s
    for _ in range(4):
        nuevo = ENTIDAD_RE.sub(lambda m: html.unescape(m.group()), s)
        if nuevo == s:
            break
        s = nuevo
        eventos['HTML_entidades'] += 1
    for malo, bueno in sorted(MOJIBAKE.items(), key=lambda p: -len(p[0])):
        if malo in s:
            eventos['Artefactos_codificacion'] += s.count(malo)
            s = s.replace(malo, bueno)
    nuevo = nfc(s)
    if nuevo != s:
        eventos['Unicode_NFC'] += 1
    s = nuevo
    nuevo = re.sub(r'<\s*(sub|sup)\s*>(.*?)<\s*/\s*\1\s*>', convertir_indice_html, s, flags=re.I | re.S)
    if nuevo != s:
        eventos['HTML_indices_cientificos'] += 1
    s = nuevo
    s, num = re.subn(r'<\s*br\s*/?\s*>|</?\s*(?:p|div|li|ul|ol|h[1-6])\b[^>]*>', ' ', s, flags=re.I)
    s, num2 = re.subn(r'</?\s*(?:i|em|b|strong|span|a|u|small)\b[^>]*>', '', s, flags=re.I)
    if num + num2:
        eventos['HTML_etiquetas'] += num + num2
    nuevo = quitar_espacios(nfc(s))
    if nuevo != s:
        eventos['Espacios'] += 1
    return nuevo


def looks_allcaps(s):
    palabras = re.findall(r'[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]{2,}', s)
    letras = [c for c in s if c.isalpha()]
    return len(palabras) >= 3 and bool(letras) and sum(c.isupper() for c in letras) / len(letras) >= .90

# ============================================================
# 3. LECTURA Y ESCRITURA SEGURA DE CSV
# ============================================================


def sha256_archivo(ruta):
    return hashlib.sha256(Path(ruta).read_bytes()).hexdigest()


def leer_csv_texto(ruta, columnas=None):
    ruta = Path(ruta)
    with ruta.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f)
        try:
            encabezado = next(reader)
        except StopIteration:
            raise ValueError(f'{ruta.name}: archivo vacío, ni siquiera tiene encabezado.')
        if len(encabezado) != len(set(encabezado)) or any(not x or x.startswith('Unnamed') for x in encabezado):
            raise ValueError(f'{ruta.name}: encabezados duplicados, vacíos o Unnamed; revisar el insumo sin repararlo silenciosamente.')
        if columnas is not None and encabezado != columnas:
            raise ValueError(f'{ruta.name}: esquema inesperado: {encabezado}')
        for numero, registro in enumerate(reader, 2):
            if len(registro) != len(encabezado):
                raise ValueError(f'{ruta.name}: registro CSV {numero} con {len(registro)} campos, se esperaban {len(encabezado)}.')
    datos = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    if columnas is not None and list(datos.columns) != columnas:
        raise ValueError('No coincide el esquema leído.')
    if datos.isna().any().any():
        raise ValueError(f'{ruta.name}: se detectaron valores no textuales.')
    return datos


def validar_no_notacion_cientifica(datos):
    problemas = []
    for campo in ('ISBN', 'ISSN'):
        for idx, val in datos[campo].items():
            if SCI_RE.search(val):
                problemas.append((int(idx) + 1, campo, val))
    if problemas:
        raise ValueError('DETENIDO: notación científica en la entrada. No se reconstruyen dígitos ni se toma una base histórica. ' + repr(problemas[:15]))


def preparar_salidas(productos, insumos):
    # Comprobar TODOS los destinos antes de escribir cualquiera.
    insumos = {Path(p).resolve() for p in insumos}
    for ruta, tabla in productos:
        ruta = Path(ruta)
        if ruta.resolve() in insumos:
            raise ValueError('Una salida intentaría sobrescribir un insumo.')
        if any(not isinstance(v, str) for v in tabla.to_numpy().ravel()):
            raise TypeError(f'{ruta.name}: la tabla a guardar contiene valores que no son texto.')
        if ruta.exists() and not actualizar_archivos:
            try:
                existente = leer_csv_texto(ruta, list(tabla.columns))
            except Exception as exc:
                raise FileExistsError(f'{ruta.name}: existe una salida diferente o inválida. Resguárdala antes de activar actualizar_archivos=True.') from exc
            if not existente.equals(tabla.reset_index(drop=True)):
                raise FileExistsError(f'{ruta.name}: existe una salida diferente; no se sobrescribió. Resguárdala o autoriza actualizar_archivos=True.')


def guardar_csv_seguro(tabla, ruta):
    ruta = Path(ruta)
    tabla = tabla.reset_index(drop=True)
    if ruta.exists() and not actualizar_archivos:
        lectura = leer_csv_texto(ruta, list(tabla.columns))
        if lectura.equals(tabla):
            return
        raise FileExistsError(f'{ruta.name}: no se autorizó sobrescritura.')
    ruta.parent.mkdir(parents=True, exist_ok=True)
    temporal = ruta.with_name(ruta.name + '.__tmp__')
    if temporal.exists():
        raise FileExistsError(f'Existe el temporal {temporal}; revisar antes de continuar.')
    try:
        tabla.to_csv(temporal, index=False, encoding='utf-8-sig', quoting=csv.QUOTE_ALL, lineterminator='\n')
        relectura = leer_csv_texto(temporal, list(tabla.columns))
        if not relectura.equals(tabla):
            raise ValueError(f'{ruta.name}: cambió el contenido al guardar/releer.')
        os.replace(temporal, ruta)
        if not leer_csv_texto(ruta, list(tabla.columns)).equals(tabla):
            raise ValueError(f'{ruta.name}: la salida definitiva no coincide con memoria.')
    finally:
        if temporal.exists():
            temporal.unlink()

# ============================================================
# 4. DECISIONES MANUALES: ALCANCE, ORIGINAL Y CONFLICTOS
# ============================================================


def preparar_revision(revision):
    if list(revision.columns) != COLUMNAS_REVISION:
        raise ValueError('La revisión no tiene el esquema de esta fase; no utilizar automáticamente el CSV histórico de 753 decisiones.')
    activos = []
    for numero, r in revision.iterrows():
        if r['Aplicar'] not in {'SI', 'NO', 'REVISAR'}:
            raise ValueError(f'Revisión {numero+1}: Aplicar inválido.')
        if r['Aplicar'] != 'SI':
            continue
        if r['Campo'] not in MODIFICABLES or r['Campo'] == 'SubArea':
            raise ValueError(f'Revisión {numero+1}: columna protegida o fuera de alcance.')
        if r['Decision_manual'] not in {'CORREGIR', 'CONSERVAR', 'ELIMINAR'}:
            raise ValueError(f'Revisión {numero+1}: decisión inválida.')
        if not r['Comentario_resolucion'].strip() or not r['Valor_original'].strip():
            raise ValueError(f'Revisión {numero+1}: falta original o justificación; no se admiten instrucciones de completado.')
        alcance = r['Alcance']
        if alcance not in {'CELDA', 'ELEMENTO', 'COLA', 'SEPARADOR'}:
            raise ValueError(f'Revisión {numero+1}: alcance inválido.')
        if alcance == 'CELDA' and (not r['Fuente_origen'] or not r['Titulo']):
            raise ValueError(f'Revisión {numero+1}: corrección de celda sin contexto de fuente/publicación.')
        if alcance in {'ELEMENTO', 'SEPARADOR'} and r['Campo'] != 'Keywords':
            raise ValueError('ELEMENTO y SEPARADOR solo se admiten en Keywords.')
        if alcance == 'COLA' and (r['Campo'] != 'Abstract' or r['Decision_manual'] != 'ELIMINAR' or r['Valor_final']):
            raise ValueError('Una COLA solo puede retirar el fragmento terminal exacto de un Abstract.')
        if alcance == 'ELEMENTO' and r['Decision_manual'] == 'ELIMINAR' and r['Valor_final']:
            raise ValueError('Eliminar un elemento no puede introducir un sustituto.')
        if alcance == 'CELDA' and r['Campo'] in {'Titulo', 'Abstract'} and r['Decision_manual'] == 'ELIMINAR':
            raise ValueError('No se elimina una celda científica desde una regla manual genérica. Los placeholders se atienden con la regla de ausencia.')
        d = r.to_dict()
        d['_id'] = 'M' + str(numero+1).zfill(4)
        d['_original'] = texto_clave(d['Valor_original'])
        activos.append(d)
    return activos


def coincide_contexto(r, fila):
    for campo in ('Fuente_origen', 'indice'):
        if r[campo] and r[campo] != fila[campo]:
            return False
    if r['Titulo'] and texto_clave(r['Titulo']) != texto_clave(fila['Titulo']):
        return False
    if r['Doi'] and doi_clave(r['Doi']) != doi_clave(fila['Doi']):
        return False
    return True


def aplicar_celda(original, campo, fila, decisiones, eventos, usados):
    reglas = [r for r in decisiones if r['Campo'] == campo and r['Alcance'] == 'CELDA'
              and r['_original'] == texto_clave(original) and coincide_contexto(r, fila)]
    finales = {r['Valor_final'] if r['Decision_manual'] == 'CORREGIR' else original for r in reglas}
    if len(finales) > 1:
        raise ValueError(f'Decisiones manuales contradictorias en {campo}: {[r["_id"] for r in reglas]}')
    if reglas:
        for r in reglas:
            usados.add(r['_id'])
        eventos['Decision_manual_celda'] += 1
        resultado = next(iter(finales))
        if not original.strip() and resultado.strip():
            raise ValueError('Una decisión intentó completar un vacío.')
        if campo == 'Abstract' and resultado != original:
            # Las correcciones de texto de esta entrega son locales; no se acepta
            # sustituir el abstract completo por una versión de otra fuente.
            from difflib import SequenceMatcher
            if SequenceMatcher(None, original, resultado, autojunk=False).ratio() < .85:
                raise RevisionNecesaria('ABSTRACT_CAMBIO_MANUAL_EXTENSO', 'Revisar la decisión: no debe parafrasear ni sustituir el abstract.')
        return resultado
    return original

# ============================================================
# 5. TÍTULO Y AÑO
# ============================================================


def clean_title(s, eventos, aprobado=False):
    x = clean_text(s, eventos)
    if looks_allcaps(x) and not aprobado:
        raise RevisionNecesaria('TITULO_ALL_CAPS', 'Capitalización no resuelta: no se aplica sentence case automáticamente.')
    return x


def clean_year(s, eventos):
    x = clean_text(s, eventos)
    if not x:
        return ''
    m = re.fullmatch(r'(2024|2025)(?:\.0+)?', x)
    if not m:
        raise RevisionNecesaria('ANIO_NO_VALIDO', 'Solo se admiten 2024, 2025 o vacío; no extraer el año de DOI/URL.')
    if x != m[1]:
        eventos['Anio_formato'] += 1
    return m[1]

# ============================================================
# 6. ISBN: CHECKSUM Y SEGMENTACIÓN CON EVIDENCIA
# ============================================================


def compact_isbn(s):
    return re.sub(r'[\s-]', '', s.translate(GUIONES)).upper()


def isbn13_valid(s):
    return bool(re.fullmatch(r'97[89][0-9]{10}', s)) and sum(int(c)*(1 if i%2==0 else 3) for i,c in enumerate(s)) % 10 == 0


def isbn10_valid(s):
    return bool(re.fullmatch(r'[0-9]{9}[0-9X]', s)) and sum((10-i)*(10 if c=='X' else int(c)) for i,c in enumerate(s)) % 11 == 0


def separar_isbn_presentacion(s, fuente, doi, eventos):
    if SCI_RE.search(s):
        raise ValueError('ISBN en notación científica: detener y revisar la entrada.')
    x = clean_text(s, eventos)
    partes = []
    for token in x.split(';'):
        token = token.strip()
        if not token:
            continue
        limpio = re.sub(r'^(?:ISBN(?:-1[03])?)\s*:\s*', '', token, flags=re.I)
        # Se admite también ISBN-10/13 seguido de espacio, sin dos puntos.
        limpio = re.sub(r'^ISBN(?:-1[03])?\s+', '', limpio, flags=re.I).strip()
        m = re.fullmatch(r'(.+?)/([0-9]{2,4})/([0-9]{2})', limpio)
        if m and (fuente == 'ACM' or doi_clave(doi).startswith('10.1145/')):
            candidato = compact_isbn(m[1])
            if isbn13_valid(candidato) or isbn10_valid(candidato):
                limpio = m[1]
                eventos['ISBN_sufijo_presentacion'] += 1
        comp = compact_isbn(limpio)
        if not (isbn13_valid(comp) or isbn10_valid(comp)):
            raise RevisionNecesaria('ISBN_INVALIDO', f'ISBN con caracteres, longitud o checksum no válidos: {token!r}. No se borran ni inventan dígitos.')
        if limpio != token:
            eventos['ISBN_prefijo_o_sufijo'] += 1
        partes.append(comp)
    return partes


def preparar_segmentacion(tabla):
    necesarios = {'ISBN_compacto', 'ISBN_segmentado', 'Prefijo', 'Grupo_registral',
                  'Registrante', 'Rango_registrante', 'Fecha_rangos', 'Serial_rangos',
                  'Evidencia', 'URL_evidencia'}
    if set(tabla.columns) != necesarios:
        raise ValueError('Tabla de segmentación ISBN inesperada.')
    mapa = {}
    for _, r in tabla.iterrows():
        comp, form = r['ISBN_compacto'], r['ISBN_segmentado']
        if not (isbn13_valid(comp) or isbn10_valid(comp)) or compact_isbn(form) != comp:
            raise ValueError('La tabla ISBN altera dígitos o contiene un checksum incorrecto.')
        if comp in mapa:
            raise ValueError('La tabla ISBN contiene identificadores repetidos.')
        p = form.split('-')
        prefijo = p[0] if len(comp)==13 else '978'
        grupo, registrante, publicacion, control = p[1:] if len(comp)==13 else p
        minimo, maximo = r['Rango_registrante'].split('-')
        if (prefijo != r['Prefijo'] or grupo != r['Grupo_registral'] or registrante != r['Registrante']
            or not (len(registrante)==len(minimo)==len(maximo) and minimo<=registrante<=maximo)
            or not publicacion.isdigit() or not r['URL_evidencia'] or not r['Fecha_rangos']):
            raise ValueError('Segmentación ISBN sin rango o evidencia consistente.')
        mapa[comp] = form
    return mapa


def clean_isbn_cell(s, fuente, doi, mapa, eventos):
    if not s.strip():
        return ''
    salida = []
    vistos = set()
    for comp in separar_isbn_presentacion(s, fuente, doi, eventos):
        if comp in vistos:
            eventos['ISBN_duplicados_internos'] += 1
            continue
        if comp not in mapa:
            raise RevisionNecesaria('ISBN_SEGMENTACION_PENDIENTE', f'ISBN válido {comp}, pero falta segmentación documentada. No usar una máscara fija; añadir evidencia a segmentacion_isbn_validada.csv.')
        salida.append(mapa[comp])
        vistos.add(comp)
    return '; '.join(salida)

# ============================================================
# 7. ISSN
# ============================================================


def issn_valid(s):
    return bool(re.fullmatch(r'[0-9]{7}[0-9X]', s)) and sum((8-i)*(10 if c=='X' else int(c)) for i,c in enumerate(s)) % 11 == 0


def clean_issn_cell(s, eventos):
    x = clean_text(s, eventos)
    if not x:
        return ''
    if SCI_RE.search(x):
        raise ValueError('ISSN en notación científica: detener y revisar la entrada.')
    salida, vistos = [], set()
    for token in x.split(';'):
        token = token.strip()
        if not token:
            continue
        limpio = re.sub(r'^(?:e-?ISSN|p-?ISSN|ISSN)\s*:?\s*', '', token, flags=re.I)
        comp = re.sub(r'[\s-]', '', limpio.translate(GUIONES)).upper()
        if not issn_valid(comp):
            raise RevisionNecesaria('ISSN_INVALIDO', f'ISSN inválido: {token!r}; no completar ceros ni reemplazar dígitos.')
        if comp in vistos:
            eventos['ISSN_duplicados_internos'] += 1
            continue
        vistos.add(comp)
        salida.append(comp[:4] + '-' + comp[4:])
    return '; '.join(salida)

# ============================================================
# 8. DOI Y URL
# ============================================================


def clean_doi(s, eventos):
    x = clean_text(s, eventos)
    if not x:
        return ''
    limpio = doi_clave(x)
    if limpio != x:
        eventos['DOI_wrappers_o_caja'] += 1
    if not DOI_RE.fullmatch(limpio):
        raise RevisionNecesaria('DOI_SINTAXIS', 'El DOI no tiene sintaxis limpia. No se reconstruye ni se copia de otra fila.')
    # La sintaxis no equivale a verificar que el DOI esté registrado o corresponda al artículo.
    return limpio


def url_sintactica(x):
    if not x:
        return True
    if re.search(r'[\s\x00-\x1f\x7f<>]', x):
        return False
    try:
        u = urlsplit(x)
        host = u.hostname
        if u.scheme.lower() not in {'http', 'https'} or not host or u.username or u.password:
            return False
        _ = u.port
        dominio = host.encode('idna').decode('ascii')
        if ':' not in dominio:  # IPv6 ya es validado por urlsplit.
            if '.' not in dominio or any(not re.fullmatch(r'[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?', p) for p in dominio.rstrip('.').split('.')):
                return False
        return not bool(re.search(r'%(?![0-9A-Fa-f]{2})', x))
    except (ValueError, UnicodeError):
        return False


def clean_url(s, eventos):
    # No lower() sobre URL completa: ruta, parámetros y fragmento pueden distinguir caja.
    x = clean_text(s, eventos)
    sin_espacios = re.sub(r'\s+', '', x)
    if sin_espacios != x:
        eventos['URL_espacios'] += 1
    if not url_sintactica(sin_espacios):
        raise RevisionNecesaria('URL_SINTAXIS', 'Revisar esquema http/https, dominio y caracteres; no sustituir por un enlace DOI.')
    return sin_espacios


def url_es_perfil(x):
    if not x:
        return False
    u = urlsplit(x)
    host, ruta = (u.hostname or '').lower(), u.path.lower()
    return (host in {'orcid.org','www.orcid.org'}
            or ('scholar.google' in host and ruta.startswith('/citations'))
            or re.search(r'/(?:profile|profiles|author|authors|people|researchers?)/', ruta) is not None
            or bool(re.search(r'/(?:cv|curriculum)(?:\.|/|$)|/~[^/]+/?$', ruta)))

# ============================================================
# 9. KEYWORDS
# ============================================================


def separar_nivel_superior(s, delimitador):
    # No separar comas o punto y coma dentro de paréntesis/corchetes/llaves.
    resultado, actual, pila = [], [], []
    parejas = {')':'(', ']':'[', '}':'{'}
    i = 0
    while i < len(s):
        c = s[i]
        if c in '([{':
            pila.append(c)
        elif c in ')]}' and pila and pila[-1] == parejas[c]:
            pila.pop()
        if not pila and s.startswith(delimitador, i):
            resultado.append(''.join(actual).strip())
            actual = []
            i += len(delimitador)
            continue
        actual.append(c)
        i += 1
    resultado.append(''.join(actual).strip())
    return [p for p in resultado if p]


def parse_keywords(s, fila, decisiones, eventos, usados):
    s = clean_text(s, eventos)
    if not s:
        return []
    reglas = [r for r in decisiones if r['Campo']=='Keywords' and r['Alcance']=='SEPARADOR'
              and r['_original']==texto_clave(s) and coincide_contexto(r,fila)]
    if reglas:
        finales = {r['Valor_final'] for r in reglas}
        if len(finales)>1:
            raise ValueError('Decisiones contradictorias para separadores de keywords.')
        esperado = '; '.join(separar_nivel_superior(s, ','))
        elegido = next(iter(finales))
        if texto_clave(esperado) != texto_clave(elegido):
            raise ValueError('Una regla de separadores añade, elimina o sustituye palabras.')
        s = elegido
        usados.update(r['_id'] for r in reglas)
        eventos['Keywords_separador_coma_revisado'] += 1
    elif ';' not in s and ',' in s:
        # Solo se aceptan las listas con comas realmente inspeccionadas.
        raise RevisionNecesaria('KEYWORDS_SEPARADOR_AMBIGUO', 'Comas sin regla de separación revisada: no fragmentar automáticamente términos compuestos.')
    bloques = separar_nivel_superior(s, ';')
    if fila['Fuente_origen'] == 'EV':
        items = []
        for bloque in bloques:
            mascaras = {}
            for frase in EV_HYPHEN_EXCEPTIONS:
                if frase in bloque:
                    llave = '\ue000' + str(len(mascaras)) + '\ue001'
                    mascaras[llave] = frase
                    bloque = bloque.replace(frase, llave)
            for token in separar_nivel_superior(bloque, ' - '):
                for llave, frase in mascaras.items():
                    token = token.replace(llave, frase)
                items.append(token)
        if len(items) != len(bloques):
            eventos['Keywords_separador_EV'] += 1
        return items
    return bloques


def capitalizar_keyword(x, eventos):
    original = x
    if x in GENERIC_ALLCAPS_SINGLE or x in GENERIC_ALLCAPS_PHRASES:
        x = x[0] + x[1:].lower()
    # Corregir solo grafías técnicas de la lista cerrada, como pH o CNN.
    def tecnica(m):
        t = m.group()
        return TECNICAS.get(t.casefold(), t)
    x = re.sub(r'(?<![\w])(?:' + '|'.join(re.escape(v) for v in GRAFIAS_TECNICAS) + r')(?![\w])', tecnica, x, flags=re.I)
    m = re.match(r'([A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+)', x)
    if m:
        palabra = m[1]
        if (palabra.islower() and len(palabra)>1 and palabra.casefold() not in TECNICAS):
            # No tocar el resto: deep learning -> Deep learning, nunca Deep Learning.
            x = x[:1].upper() + x[1:]
    if original != x:
        eventos['Keywords_capitalizacion'] += 1
    return x


def clean_keywords_cell(s, fila, decisiones, eventos, usados):
    salida, vistos = [], set()
    for item in parse_keywords(s, fila, decisiones, eventos, usados):
        x = clean_text(item, eventos)
        reglas = [r for r in decisiones if r['Campo']=='Keywords' and r['Alcance']=='ELEMENTO'
                  and r['_original'].casefold()==texto_clave(x).casefold() and coincide_contexto(r,fila)]
        finales = {'' if r['Decision_manual']=='ELIMINAR' else r['Valor_final'] for r in reglas}
        if len(finales)>1:
            raise ValueError('Decisiones manuales contradictorias para un keyword.')
        if reglas:
            usados.update(r['_id'] for r in reglas)
            eventos['Keywords_elementos_manual'] += 1
            x = next(iter(finales))
            if not x:
                eventos['Keywords_residuos_eliminados'] += 1
                continue
        x = x.rstrip(',').strip()
        if not x:
            continue
        x = capitalizar_keyword(x, eventos)
        clave = nfc(x).casefold()
        if clave in vistos:
            eventos['Keywords_duplicados_internos'] += 1
            continue
        vistos.add(clave)
        salida.append(x)
    return '; '.join(salida)

# ============================================================
# 10. ABSTRACT
# ============================================================


def clean_abstract(s, fila, decisiones, eventos, usados):
    x = clean_text(s, eventos)
    if not x:
        return ''
    x, n = PREFIJO_ABSTRACT_RE.subn('', x, count=1)
    if n:
        eventos['Abstract_prefijos_eliminados'] += 1
    if x.strip().casefold() in AUSENCIA_ABSTRACT:
        eventos['Abstract_placeholders_eliminados'] += 1
        return ''
    colas = []
    for r in decisiones:
        if r['Campo'] != 'Abstract' or r['Alcance'] != 'COLA' or not coincide_contexto(r, fila):
            continue
        cola = clean_text(r['Valor_original'], Counter())
        if cola and x.endswith(cola) and len(x)>len(cola):
            inicio = len(x)-len(cola)
            prefijo = x[:inicio].rstrip()
            frontera_clara = bool(prefijo) and (prefijo[-1] in '.!?' or bool(r['Titulo']))
            if x[inicio-1].isspace() and frontera_clara:
                colas.append((len(cola), inicio, r))
    if colas:
        # El más largo evita dejar "Copyright" si el sufijo interno comienza en ©.
        _, inicio, r = max(colas, key=lambda z:z[0])
        x = x[:inicio].rstrip()
        usados.add(r['_id'])
        eventos['Abstract_colas_eliminadas'] += 1
        if 'LISTA_AUTORES' in r['Problema']:
            eventos['Abstract_colas_autores'] += 1
        elif 'AGRADECIMIENTO' in r['Problema']:
            eventos['Abstract_colas_agradecimientos'] += 1
        else:
            eventos['Abstract_copyright_eliminado'] += 1
            if 'GRAFICO' in r['Problema']:
                eventos['Abstract_marcadores_graficos_eliminados'] += 1
    # No recortar desde una palabra suelta copyright/author/By.
    # Las colas desconocidas se señalan para revisión, sin eliminarlas.
    return quitar_espacios(nfc(x))

# ============================================================
# 11. LIMPIEZA INDEPENDIENTE POR CELDA Y AUDITORÍA
# ============================================================


def transformar(entrada, revision, tabla_segmentacion):
    if list(entrada.columns) != COLUMNAS:
        raise ValueError('La entrada debe conservar exactamente las 15 columnas, en su orden.')
    validar_no_notacion_cientifica(entrada)
    if entrada.empty:
        raise ValueError('La base de entrada no contiene registros.')
    decisiones = preparar_revision(revision)
    mapa_isbn = preparar_segmentacion(tabla_segmentacion)
    salida = entrada.copy(deep=True)
    detalles, pendientes = [], []
    totales = Counter({'Abstract_codificacion_corregida_celdas': 0, 'Artefactos_codificacion': 0,
                       'ISBN_duplicados_internos': 0, 'ISSN_duplicados_internos': 0,
                       'Abstract_colas_agradecimientos': 0})
    casos_aplicados = set()
    celdas_manuales = 0
    metodos_por_celda = {}

    def pendiente(i, fila, campo, propuesta, codigo, mensaje, severidad='ERROR'):
        pendientes.append(dict(zip(COLUMNAS_PENDIENTES, [str(i+1), fila['Fuente_origen'], fila['indice'],
            fila['Titulo'], fila['Doi'], campo, fila[campo], propuesta, codigo, severidad,
            mensaje, 'REVISAR', '', ''])))

    for i, fila in entrada.iterrows():
        if fila['Area'] not in AREAS_VALIDAS:
            pendiente(i,fila,'Area',fila['Area'],'AREA_NO_VALIDA','El área se conserva intacta; resolver fuera de esta fase o autorizar expresamente.')
        for campo in MODIFICABLES:
            original = fila[campo]
            eventos, usados = Counter(), set()
            try:
                valor = aplicar_celda(original, campo, fila, decisiones, eventos, usados)
                if campo == 'Titulo':
                    nuevo = clean_title(valor,eventos,aprobado=bool(usados))
                elif campo == 'Año':
                    nuevo = clean_year(valor,eventos)
                    if clean_year(original,Counter()) != nuevo:
                        raise ValueError('No se decide aquí un conflicto de años ni se sustituye por un año externo.')
                elif campo == 'ISBN':
                    nuevo = clean_isbn_cell(valor,fila['Fuente_origen'],fila['Doi'],mapa_isbn,eventos)
                    # Incluso una corrección manual no puede traer otros identificadores.
                    a = list(dict.fromkeys(separar_isbn_presentacion(original,fila['Fuente_origen'],fila['Doi'],Counter())))
                    b = [compact_isbn(x) for x in nuevo.split('; ') if x]
                    if a != b:
                        raise ValueError('La limpieza ISBN intentó cambiar los identificadores originales.')
                elif campo == 'ISSN':
                    nuevo = clean_issn_cell(valor,eventos)
                    if clean_issn_cell(original,Counter()) != nuevo:
                        raise ValueError('La revisión ISSN intentó cambiar identificadores en vez de formato.')
                elif campo == 'Doi':
                    nuevo = clean_doi(valor,eventos)
                    if doi_clave(clean_text(original,Counter())) != nuevo:
                        raise ValueError('La revisión DOI no es solo una limpieza de representación.')
                elif campo == 'URL':
                    nuevo = clean_url(valor,eventos)
                    if clean_url(original,Counter()) != nuevo:
                        raise ValueError('La revisión URL intentó sustituir el enlace.')
                elif campo == 'Keywords':
                    nuevo = clean_keywords_cell(valor,fila,decisiones,eventos,usados)
                elif campo == 'Abstract':
                    nuevo = clean_abstract(valor,fila,decisiones,eventos,usados)
                else:
                    nuevo = ''
                    if original:
                        eventos['SubArea_vaciada_regla_vigente'] += 1
                if not original.strip() and nuevo.strip():
                    raise ValueError(f'La fase intentó completar {campo} en fila {i+1}.')
                if campo in {'Titulo','Keywords','Abstract','Doi','URL'}:
                    if ARTEFACTOS_RE.search(nuevo):
                        raise RevisionNecesaria('ARTEFACTO_NO_RESUELTO', 'Se conserva el original hasta contar con una corrección inequívoca documentada.')
                    if re.search(r'</?(?:math|mml:math|jats:[A-Za-z_-]+|[A-Za-z][A-Za-z0-9:_-]{2,})(?:\s[^<>]*)?\s*/?>',nuevo):
                        raise RevisionNecesaria('HTML_NO_INTERPRETADO','Marcado no reconocido; no eliminar fórmulas o texto científico.')
                    if ENTIDAD_RE.search(nuevo):
                        raise RevisionNecesaria('ENTIDAD_HTML_NO_RESUELTA','Entidad desconocida; no sustituirla por intuición.')
                salida.at[i,campo] = nuevo
                if campo == 'Abstract' and original != nuevo and (eventos['Artefactos_codificacion'] or any(r['_id'] in usados and 'CODIFICACION' in r['Problema'] for r in decisiones)):
                    eventos['Abstract_codificacion_corregida_celdas'] += 1
                totales.update(eventos)
                if usados:
                    casos_aplicados.update(usados)
                    celdas_manuales += 1
                if campo == 'URL' and url_es_perfil(nuevo):
                    pendiente(i,fila,campo,nuevo,'URL_PERFIL_PERSONAL','La URL se conserva. Verificar su pertinencia en la revisión/completado.', 'AVISO')
                if campo == 'Abstract':
                    # Solo marcar marcadores terminales explícitos no cubiertos.
                    tail = nuevo[-700:]
                    if re.search(r'(?:©\s*(?:[12][0-9]{3}|The Authors?)|\bCopyright\s+(?:[12][0-9]{3}|©|Author|owned)|\b(?:Corresponding author|Authors?|Acknowledg(?:e)?ments)\s*:)\s*',tail,re.I):
                        pendiente(i,fila,campo,nuevo,'ABSTRACT_COLA_POR_REVISAR','Posible texto adicional terminal no cubierto por una regla exacta. No recortar sin comprobar límites.', 'AVISO')
                    if re.search(r'(?:[;,]\s*p|[=<>]|\b(?:and|or|with|of|the))\s*$',nuevo,re.I):
                        pendiente(i,fila,campo,nuevo,'ABSTRACT_POSIBLE_TRUNCAMIENTO_ORIGEN','El texto de origen parece terminar incompleto. Se conserva lo existente; no buscar, copiar ni reconstruir el fragmento faltante en esta fase.', 'AVISO')
                if original != nuevo:
                    evidencia = []
                    for r in decisiones:
                        if r['_id'] in usados:
                            evidencia.append(r['_id'] + ': ' + r['Comentario_resolucion'] + (' | ' + r['URL_evidencia'] if r['URL_evidencia'] else ''))
                    if campo=='ISBN':
                        evidencia.append('segmentacion_isbn_validada.csv; rangos y serial registrados por ISBN; mismos dígitos y checksum.')
                    if not evidencia:
                        evidencia.append('Transformación del contenido de la propia celda; reglas del código 05, sin propagación ni búsqueda de metadatos.')
                    detalles.append(dict(zip(COLUMNAS_DETALLE,[str(i+1),fila['Fuente_origen'],fila['indice'],fila['Autor_norm'],
                        fila['Titulo'],fila['Doi'],campo,original,nuevo,'; '.join(sorted(eventos)) or 'Formato',
                        '; '.join(sorted(usados)),' || '.join(evidencia)])))
            except RevisionNecesaria as exc:
                # Un campo inválido NO se vacía ni se altera para pasar la validación.
                salida.at[i,campo] = original
                pendiente(i,fila,campo,original,exc.codigo,str(exc))

    detalle = pd.DataFrame(detalles,columns=COLUMNAS_DETALLE).fillna('').astype(str)
    casos = pd.DataFrame(pendientes,columns=COLUMNAS_PENDIENTES).fillna('').astype(str)
    metricas = [('Filas_entrada',len(entrada)),('Filas_salida',len(salida)),
                ('Columnas_entrada',len(entrada.columns)),('Columnas_salida',len(salida.columns))]
    for campo in COLUMNAS:
        metricas.append((campo+'_celdas_modificadas',int(entrada[campo].ne(salida[campo]).sum())))
        if campo in MODIFICABLES:
            metricas.append((campo+'_vacios_antes',int(entrada[campo].str.strip().eq('').sum())))
            metricas.append((campo+'_vacios_despues',int(salida[campo].str.strip().eq('').sum())))
    metricas.extend(sorted(totales.items()))
    metricas.extend([('Casos_manuales_distintos_aplicados',len(casos_aplicados)),
                    ('Celdas_con_regla_manual',celdas_manuales),
                    ('Reglas_apoyo_activas',len(decisiones)),
                    ('Reglas_apoyo_no_aplicadas',len(decisiones)-len(casos_aplicados)),
                    ('ISBN_distintos_con_segmentacion_documentada',len(mapa_isbn)),
                    ('Casos_revision_ERROR',int(casos['Severidad'].eq('ERROR').sum())),
                    ('Casos_revision_AVISO',int(casos['Severidad'].eq('AVISO').sum())),
                    ('Duplicados_exactos_entrada',int(entrada.duplicated().sum())),
                    ('Duplicados_exactos_salida',int(salida.duplicated().sum())),
                    ('Filas_eliminadas',0),('Filas_fusionadas',0),('Celdas_completadas',0)])
    resumen = pd.DataFrame([(str(k),str(v)) for k,v in metricas],columns=['Metrica','Valor'])
    return salida, resumen, detalle, casos

# ============================================================
# 12. VALIDACIONES FINALES EN MEMORIA
# ============================================================


def validar_resultado(entrada,salida,pendientes):
    if salida.shape != entrada.shape or list(salida.columns) != COLUMNAS:
        raise AssertionError('Cambió la estructura de la base.')
    if not entrada[PROTEGIDAS].equals(salida[PROTEGIDAS]):
        raise AssertionError('Cambió una columna protegida.')
    if not salida['SubArea'].eq('').all():
        raise AssertionError('SubArea no quedó vacía.')
    for campo in MODIFICABLES:
        if (entrada[campo].str.strip().eq('') & salida[campo].str.strip().ne('')).any():
            raise AssertionError('Se completó un vacío.')
    validar_no_notacion_cientifica(salida)
    if pendientes['Severidad'].eq('ERROR').any():
        return False
    if not salida['Año'].isin(['','2024','2025']).all():
        raise AssertionError('Años inválidos.')
    for val in salida['ISBN']:
        for token in val.split('; '):
            if token and not (isbn13_valid(compact_isbn(token)) or isbn10_valid(compact_isbn(token))):
                raise AssertionError('ISBN inválido en salida.')
    for val in salida['ISSN']:
        for token in val.split('; '):
            if token and (not re.fullmatch(r'[0-9]{4}-[0-9]{3}[0-9X]',token) or not issn_valid(token.replace('-',''))):
                raise AssertionError('ISSN inválido en salida.')
    if not salida['Doi'].map(lambda v:not v or bool(DOI_RE.fullmatch(v))).all():
        raise AssertionError('DOI sin formato limpio.')
    if not salida['URL'].map(url_sintactica).all():
        raise AssertionError('URL con error sintáctico.')
    for val in salida['Keywords']:
        partes = separar_nivel_superior(val,';')
        claves = [nfc(x).casefold() for x in partes]
        if len(claves)!=len(set(claves)):
            raise AssertionError('Keywords repetidas dentro de una celda.')
        if '; '.join(partes)!=val:
            raise AssertionError('Separador Keywords inconsistente.')
    if salida['Abstract'].str.casefold().isin(AUSENCIA_ABSTRACT).any():
        raise AssertionError('Quedó un marcador de ausencia en Abstract.')
    for c in ('Titulo','Keywords','Abstract','Doi','URL'):
        if salida[c].map(lambda x:bool(ARTEFACTOS_RE.search(x))).any():
            raise AssertionError('Quedó un artefacto conocido sin resolver.')
    return True

# ============================================================
# 13. EJECUCIÓN Y ARCHIVOS DE RESULTADO
# ============================================================

entrada = leer_csv_texto(archivo_entrada,COLUMNAS)
hash_entrada = sha256_archivo(archivo_entrada)
if sha256_entrada_esperada and hash_entrada != sha256_entrada_esperada:
    raise ValueError('La entrada no corresponde a la huella esperada.')
validar_no_notacion_cientifica(entrada)
revision = leer_csv_texto(archivo_revision,COLUMNAS_REVISION)
segmentacion = leer_csv_texto(archivo_segmentacion)
hash_revision = sha256_archivo(archivo_revision)
hash_segmentacion = sha256_archivo(archivo_segmentacion)

salida, auditoria, auditoria_detallada, casos_revision = transformar(entrada,revision,segmentacion)
valido = validar_resultado(entrada,salida,casos_revision)
auditoria = pd.concat([auditoria,pd.DataFrame([
    ['SHA256_entrada',hash_entrada],['SHA256_revision',hash_revision],
    ['SHA256_segmentacion',hash_segmentacion],['Validaciones_estrictas_OK',str(valido)],
    ['Estado','LIMPIEZA_VALIDADA' if valido else 'PENDIENTE_REVISION_NO_GENERAR_BASE_FINAL'],
    ['Verificacion_DOI','Sintactica; no se consulto registro ni correspondencia editorial'],
    ['Verificacion_URL','Sintactica; no se comprobaron HTTP ni disponibilidad en linea'],
    ['Evidencia_ISBN','Segmentacion documentada; no se verifico asignacion a la publicacion']
],columns=['Metrica','Valor'])],ignore_index=True)

productos = [(archivo_pendientes,casos_revision),(archivo_auditoria,auditoria),
             (archivo_detalle,auditoria_detallada)]
if valido:
    productos.append((archivo_salida,salida))
preparar_salidas(productos,[archivo_entrada,archivo_revision,archivo_segmentacion])
for ruta,tabla in productos:
    guardar_csv_seguro(tabla,ruta)
if sha256_archivo(archivo_entrada)!=hash_entrada or sha256_archivo(archivo_revision)!=hash_revision or sha256_archivo(archivo_segmentacion)!=hash_segmentacion:
    raise AssertionError('Se modificó un insumo durante la ejecución.')

# ============================================================
# 14. RESUMEN
# ============================================================

print('=== LIMPIEZA BIBLIOGRÁFICA ===')
print('Entrada:',archivo_entrada)
print('Filas:',len(entrada),'->',len(salida),'| Columnas:',len(salida.columns))
print('Columnas protegidas: idénticas a la entrada')
print('Celdas completadas: 0 | Filas eliminadas o fusionadas: 0')
print(auditoria[auditoria['Metrica'].str.endswith('_celdas_modificadas')].to_string(index=False))
print('Errores pendientes:',int(casos_revision['Severidad'].eq('ERROR').sum()))
print('Avisos que conservan el valor:',int(casos_revision['Severidad'].eq('AVISO').sum()))
print('Relectura exacta de todos los CSV guardados: OK')
if valido:
    print('Base limpia:',archivo_salida)
else:
    print('NO se generó ni actualizó autores_unam_limpios.csv. Resolver los errores de casos_revision_bibliografica.csv y documentar la decisión en el archivo de revisión resuelta.')
    if archivo_salida.exists():
        print('ATENCIÓN: existe una base limpia de una ejecución anterior; NO es una salida nueva validada.')


=== LIMPIEZA BIBLIOGRÁFICA ===
Entrada: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\04_Limpieza\02_normalizacion\autores_unam_normalizados.csv
Filas: 5106 -> 5106 | Columnas: 15
Columnas protegidas: idénticas a la entrada
Celdas completadas: 0 | Filas eliminadas o fusionadas: 0
                         Metrica Valor
Fuente_origen_celdas_modificadas     0
       indice_celdas_modificadas     0
       Titulo_celdas_modificadas    29
          Año_celdas_modificadas   540
   Autor_norm_celdas_modificadas     0
  Afiliacion1_celdas_modificadas     0
  Afiliacion2_celdas_modificadas     0
         ISBN_celdas_modificadas  1343
         ISSN_celdas_modificadas  2458
          Doi_celdas_modificadas  1400
          URL_celdas_modificadas     0
         Area_celdas_modificadas     0
      SubArea_celdas_modificadas     0
     Keywords_celdas_modificadas  3063
     Abstract_celdas_modificadas  3033
Errores pendientes: 0
Avisos que conservan el valor: 2
Relectura exacta de todos los CSV gu